# 002 · Configuring Logging

> **Standard Library › Logging & Config** · Python 3.14

Beyond `basicConfig`, `logging` is a small pipeline of **loggers, handlers, formatters and filters** that you can wire up in code or declaratively with `dictConfig`. This notebook shows how the pieces fit, how to send logs to the console and to (rotating) files, how to add context, and how to emit structured JSON.

**In this notebook you will learn:**

- the Logger → Handler → Formatter → Filter pipeline, compared with providers and filters in `Microsoft.Extensions.Logging`
- `StreamHandler`, `FileHandler` and `RotatingFileHandler`
- format strings, `datefmt` and the useful `LogRecord` attributes
- propagation and the "every message appears twice" gotcha
- `dictConfig` as the `appsettings.json` `"Logging"` section
- adding context with `extra`, filters and `LoggerAdapter`; a minimal JSON formatter

## .NET mental model

| Microsoft.Extensions.Logging / Serilog | Python `logging` |
|---|---|
| `ILogger` (category) | `Logger` (named, hierarchical) |
| Logging provider / Serilog sink (`Console`, `File`, …) | `Handler` (`StreamHandler`, `FileHandler`, …) |
| Console formatter / output template | `Formatter` |
| `AddFilter(...)`, Serilog `Filter.ByExcluding` | `Filter` objects (on a logger or a handler), plus levels |
| `"Logging": { "LogLevel": {...} }` in `appsettings.json` | `logging.config.dictConfig({...})` |
| `logger.BeginScope(...)` / Serilog enrichers | `extra=`, `LoggerAdapter`, filters that add attributes |
| Serilog `RollingFile` / `rollingInterval` | `RotatingFileHandler` / `TimedRotatingFileHandler` |

The flow of one call such as `logger.info("hi")`:

```
logger.info("hi")
  │  level check on the logger (its effective level)  ─ too low? stop
  │  logger filters                                   ─ any returns False? stop
  ▼
LogRecord ──► each handler of this logger ──► handler level + handler filters ──► formatter ──► output
  │
  └── propagate=True? ──► repeat with the parent logger's handlers (up to root)
```

## Building the pipeline by hand

```csharp
// C#
using var factory = LoggerFactory.Create(b => b
    .AddSimpleConsole(o => o.TimestampFormat = "HH:mm:ss ")
    .SetMinimumLevel(LogLevel.Debug));
var logger = factory.CreateLogger("shop");
```

In Python you create a handler, give it a formatter (and optionally a level), and attach it to a logger. We log to `sys.stdout` so output appears inline in the notebook.

In [ ]:
import logging
import sys

logger = logging.getLogger("shop")
logger.setLevel(logging.DEBUG)                  # the logger's own threshold

console = logging.StreamHandler(sys.stdout)
console.setLevel(logging.INFO)                  # the handler's threshold (can be stricter)
console.setFormatter(logging.Formatter("%(levelname)-7s %(name)s: %(message)s"))
logger.addHandler(console)

logger.debug("dropped by the console handler's level")
logger.info("order received")
logger.warning("stock is low")

## Format strings and `LogRecord` attributes

The format string uses `%(attribute)s` placeholders for fields of the `LogRecord`:

| Attribute | Meaning |
|---|---|
| `%(asctime)s` | time, formatted with `datefmt` |
| `%(levelname)s` / `%(levelno)d` | level name / number |
| `%(name)s` | logger name (the category) |
| `%(message)s` | the formatted message |
| `%(module)s`, `%(funcName)s`, `%(lineno)d` | where the call came from |
| `%(process)d`, `%(thread)d`, `%(threadName)s`, `%(taskName)s` | process / thread / asyncio task (3.12+) |

`datefmt` takes `time.strftime` codes. To make the output reproducible, we format a hand-made record with a fixed timestamp and UTC time (`converter = time.gmtime`):

In [ ]:
import time

fmt = logging.Formatter(
    fmt="%(asctime)s.%(msecs)03d %(levelname)s [%(name)s] %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%S",
)
fmt.converter = time.gmtime                     # timestamps in UTC instead of local time

record = logging.makeLogRecord({
    "name": "shop.orders", "levelno": logging.INFO, "levelname": "INFO",
    "msg": "order %s shipped", "args": (1042,),
    "created": 1767225600.250, "msecs": 250,      # 2026-01-01 00:00:00.250 UTC
})
print(fmt.format(record))

**Tip:** you can use `{}`-style templates in the *format string* with `style="{"`, e.g. `Formatter("{levelname} {name}: {message}", style="{")`. This only affects the formatter — the message arguments in `logger.info(...)` still use `%s`.

## Logging to files

`FileHandler(path)` appends to a file. Always pass `encoding="utf-8"`. `delay=True` postpones opening the file until the first record is written.

In [ ]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()
log_dir = Path(tmp.name)

file_handler = logging.FileHandler(log_dir / "shop.log", encoding="utf-8")
file_handler.setLevel(logging.DEBUG)            # the file gets everything
file_handler.setFormatter(logging.Formatter("%(levelname)s|%(name)s|%(funcName)s|%(message)s"))
logger.addHandler(file_handler)

def checkout(order_id: int) -> None:
    logger.debug("validating basket for %s", order_id)
    logger.info("checkout complete for %s", order_id)

checkout(7)                                     # console shows INFO only
file_handler.flush()
print("--- shop.log ---")
print((log_dir / "shop.log").read_text(encoding="utf-8"), end="")

### Rotating files

`RotatingFileHandler` rolls over when the file exceeds `maxBytes`, keeping `backupCount` old files (`app.log.1`, `app.log.2`, …) — like Serilog's `fileSizeLimitBytes` + `rollOnFileSizeLimit`. `TimedRotatingFileHandler` rotates by time (`when="midnight"`, …), like `rollingInterval: Day`.

In [ ]:
from logging.handlers import RotatingFileHandler

rot_logger = logging.getLogger("sensor")
rot_logger.setLevel(logging.INFO)
rot_logger.propagate = False                    # keep these out of other handlers (see below)

rotating = RotatingFileHandler(log_dir / "sensor.log", maxBytes=200, backupCount=2, encoding="utf-8")
rotating.setFormatter(logging.Formatter("%(message)s"))
rot_logger.addHandler(rotating)

for i in range(20):
    rot_logger.info("reading %02d: temperature=21.%d", i, i % 10)

for p in sorted(log_dir.glob("sensor.log*")):
    lines = p.read_text(encoding="utf-8").splitlines()
    print(f"{p.name:<13} {len(lines)} lines, first={lines[0]!r}")

Only the newest data survives: older backups beyond `backupCount` are deleted.

## Propagation and duplicate messages

⚠️ **Gotcha:** records propagate **up** the hierarchy. If you attach a handler to `shop` **and** one to the root logger, every `shop` message is emitted by both. This is the most common "why is every log line printed twice?" bug — often caused by calling a setup function (that adds a handler) more than once, e.g. re-running a notebook cell.

In [ ]:
root = logging.getLogger()
root_handler = logging.StreamHandler(sys.stdout)
root_handler.setFormatter(logging.Formatter("ROOT    %(name)s: %(message)s"))
root.addHandler(root_handler)

logger.warning("payment retry")      # printed by the shop handler AND the root handler

Fixes, in order of preference:

1. **Attach handlers only to the root logger** (or one top-level app logger); child loggers just set levels.
2. Set `logger.propagate = False` on loggers that have their own handlers.
3. Make setup idempotent (check `logger.handlers` before adding, or use `dictConfig`, which replaces configuration).

In [ ]:
logger.propagate = False
logger.warning("payment retry (once)")
logger.propagate = True               # restore for the next sections
root.removeHandler(root_handler)

## Filters

A filter decides whether a record passes (and may modify it). It can be a `logging.Filter` subclass **or any callable** taking a record. Since 3.12 a filter may also return a (new) `LogRecord` to replace the record. Attach filters to a logger (applies before handlers) or to a handler (applies only to that output).

In [ ]:
def drop_health_checks(record: logging.LogRecord) -> bool:
    return "/health" not in record.getMessage()

console.addFilter(drop_health_checks)
logger.info("GET /health 200")       # filtered out
logger.info("GET /orders/7 200")     # shown
console.removeFilter(drop_health_checks)

## Adding context: `extra`, filters and `LoggerAdapter`

`ILogger.BeginScope(...)` and Serilog enrichers attach properties such as a request id. In Python:

- `extra={"key": value}` adds attributes to **one** record — usable as `%(key)s` in format strings.
- A **filter** can add attributes to *every* record (e.g. from a `contextvars.ContextVar`).
- A **`LoggerAdapter`** wraps a logger and adds the same `extra` to every call.

⚠️ **Gotcha:** if a format string references `%(request_id)s` and a record lacks that attribute, formatting fails (printed as a "--- Logging error ---" to stderr). Give the formatter `defaults=` (3.10+) for optional fields.

In [ ]:
ctx_handler = logging.StreamHandler(sys.stdout)
ctx_handler.setFormatter(logging.Formatter(
    "%(levelname)s [req=%(request_id)s] %(message)s",
    defaults={"request_id": "-"},
))
api = logging.getLogger("api")
api.propagate = False
api.addHandler(ctx_handler)

api.warning("no context here")
api.warning("slow query", extra={"request_id": "r-001"})

In [ ]:
from contextvars import ContextVar

current_request: ContextVar[str] = ContextVar("current_request", default="-")

class RequestIdFilter(logging.Filter):
    def filter(self, record: logging.LogRecord) -> bool:
        record.request_id = current_request.get()   # enrich every record
        return True

api.addFilter(RequestIdFilter())
current_request.set("r-777")              # e.g. set by web middleware per request
api.warning("payment declined")

In [ ]:
adapter = logging.LoggerAdapter(api, extra={"request_id": "r-adapter"})
adapter.warning("via adapter")            # but our logger filter runs later and overwrites it!
api.filters.clear()
adapter.warning("via adapter, filter removed")

Note the ordering: `LoggerAdapter` puts its `extra` on the record first, then the logger's filters run — so the filter's value won. Pick **one** mechanism per attribute. (Since 3.13, `LoggerAdapter(..., merge_extra=True)` merges a call's own `extra=` with the adapter's instead of ignoring it.)

## Structured logging: a tiny JSON formatter

Serilog popularised structured logs: each event is a JSON object with properties, ready for Seq, Elasticsearch or a cloud log service. The standard library has no JSON formatter, but writing one takes a few lines — subclass `Formatter` and override `format`.

In [ ]:
import json
from datetime import datetime, timezone

class JsonFormatter(logging.Formatter):
    STANDARD = set(vars(logging.makeLogRecord({}))) | {"message", "asctime"}

    def format(self, record: logging.LogRecord) -> str:
        event = {
            "ts": datetime.fromtimestamp(record.created, tz=timezone.utc).isoformat(timespec="milliseconds"),
            "level": record.levelname,
            "logger": record.name,
            "template": record.msg,              # the un-formatted template, like Serilog's MessageTemplate
            "message": record.getMessage(),
        }
        event |= {k: v for k, v in vars(record).items() if k not in self.STANDARD}   # `extra` fields
        if record.exc_info:
            event["exception"] = self.formatException(record.exc_info)
        return json.dumps(event, default=str)

In [ ]:
json_handler = logging.StreamHandler(sys.stdout)
json_handler.setFormatter(JsonFormatter())
events = logging.getLogger("events")
events.setLevel(logging.INFO)
events.propagate = False
events.addHandler(json_handler)

events.info("order %s placed", 1042, extra={"customer": "alice", "total": 59.90})
events.warning("stock below %d", 5, extra={"sku": "APL-01"})

## Declarative setup with `dictConfig`

```json
// appsettings.json
{
  "Logging": {
    "LogLevel": { "Default": "Information", "Shop": "Debug", "Microsoft": "Warning" },
    "Console": { "FormatterName": "simple" }
  }
}
```

`logging.config.dictConfig` takes a dictionary (typically loaded from JSON, TOML or YAML) describing formatters, filters, handlers and loggers. It replaces the existing configuration, so it's safe to call again. `version: 1` is required; `disable_existing_loggers: False` keeps loggers created before the call working (the default `True` silently disables them!).

In [ ]:
import logging.config

config_json = """
{
  "version": 1,
  "disable_existing_loggers": false,
  "formatters": {
    "simple": {"format": "%(levelname)-7s %(name)s: %(message)s"},
    "json":   {"()": "__main__.JsonFormatter"}
  },
  "handlers": {
    "console": {"class": "logging.StreamHandler", "stream": "ext://sys.stdout",
                "formatter": "simple", "level": "DEBUG"}
  },
  "loggers": {
    "shop":      {"level": "DEBUG"},
    "thirdparty": {"level": "WARNING"}
  },
  "root": {"level": "INFO", "handlers": ["console"]}
}
"""
logging.config.dictConfig(json.loads(config_json))
print(logger.handlers)                        # [] — our hand-made shop handlers were replaced

logging.getLogger("shop.cart").debug("shop is DEBUG")
logging.getLogger("thirdparty.http").info("hidden: thirdparty is WARNING")
logging.getLogger("other").info("default is INFO")

⚠️ **Gotcha:** a (non-incremental) `dictConfig` call **closes every existing handler** and removes the handlers of the loggers it configures (here `shop` and the root). Loggers it doesn't mention keep their — now closed — handlers. Configure logging in **one** place, once, at application start-up.

Notable `dictConfig` features: `"ext://sys.stdout"` refers to an importable object, `"()"` names a factory (our custom `JsonFormatter`), and handlers can have `"filters"`. `logging.config.fileConfig` reads an older INI-based format — you'll see it in legacy code *(see Standard Library › Logging & Config › 003 · INI Configuration with configparser)*.

**Tip:** for high-throughput or slow outputs (network, cloud), put a `logging.handlers.QueueHandler` in front and let a `QueueListener` write from a background thread — the equivalent of Serilog's `Async` sink.

## Cleaning up

Handlers hold file handles, so close them before removing the temp directory. This is also what `logging.shutdown()` does at interpreter exit.

In [ ]:
for name in ["", "shop", "sensor", "api", "events"]:
    lg = logging.getLogger(name)
    for h in list(lg.handlers):
        lg.removeHandler(h)
        h.close()
    lg.setLevel(logging.WARNING if name == "" else logging.NOTSET)
    lg.propagate = True
    lg.filters.clear()
for name in ["thirdparty"]:
    logging.getLogger(name).setLevel(logging.NOTSET)

tmp.cleanup()
print(logging.getLogger().handlers, log_dir.exists())

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Output target | logging provider / Serilog sink | `Handler` (`StreamHandler`, `FileHandler`, …) |
| Output layout | console formatter / output template | `Formatter("%(levelname)s %(name)s: %(message)s", datefmt=...)` |
| Rolling file | Serilog `rollingInterval` / size limit | `RotatingFileHandler` / `TimedRotatingFileHandler` |
| Per-category level | `"LogLevel": {"Shop": "Debug"}` | `getLogger("shop").setLevel(logging.DEBUG)` |
| Filter records | `AddFilter(...)` | `handler.addFilter(callable)` |
| Avoid duplicates | — | handlers on root only, or `propagate = False` |
| Config from JSON | `appsettings.json` `Logging` section | `logging.config.dictConfig(dict)` |
| Scoped properties | `BeginScope` / enrichers | `extra=`, `LoggerAdapter`, enriching `Filter` |
| Structured JSON | Serilog `JsonFormatter` | custom `Formatter.format` returning `json.dumps(...)` |

## Exercises

1. Create a logger `billing` with two handlers: a `StreamHandler(sys.stdout)` that shows only `WARNING`+ with format `"%(levelname)s: %(message)s"`, and a `FileHandler` (in a temp dir) that records everything at `DEBUG`+. Log one debug, one info and one error message, then print the file content. Close the handlers afterwards.
2. Write a filter function that drops any record whose message contains `"password"`, attach it to a handler, and show that a message `"user changed password"` is not emitted.
3. Using `dictConfig`, configure the root logger with a stdout handler and format `"%(name)s -> %(message)s"` at `INFO`, then log from a logger called `jobs.nightly`. Clean up the root handlers at the end.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import logging
import logging.config
import sys
import tempfile
from pathlib import Path

# 1 — two handlers with different levels
with tempfile.TemporaryDirectory() as d:
    billing = logging.getLogger("billing")
    billing.setLevel(logging.DEBUG)
    billing.propagate = False

    out = logging.StreamHandler(sys.stdout)
    out.setLevel(logging.WARNING)
    out.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
    file_h = logging.FileHandler(Path(d) / "billing.log", encoding="utf-8")
    file_h.setFormatter(logging.Formatter("%(levelname)s %(message)s"))
    billing.addHandler(out)
    billing.addHandler(file_h)

    billing.debug("computing invoice")
    billing.info("invoice 17 created")
    billing.error("payment gateway timeout")

    for h in list(billing.handlers):
        billing.removeHandler(h)
        h.close()
    print((Path(d) / "billing.log").read_text(encoding="utf-8"), end="")

# 2 — callable filter
def no_passwords(record: logging.LogRecord) -> bool:
    return "password" not in record.getMessage().lower()

audit = logging.getLogger("audit")
audit.propagate = False
h = logging.StreamHandler(sys.stdout)
h.addFilter(no_passwords)
audit.addHandler(h)
audit.warning("user changed password")   # dropped
audit.warning("user changed email")      # shown
audit.removeHandler(h)

# 3 — dictConfig for the root logger
logging.config.dictConfig({
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {"arrow": {"format": "%(name)s -> %(message)s"}},
    "handlers": {"out": {"class": "logging.StreamHandler", "stream": "ext://sys.stdout",
                         "formatter": "arrow"}},
    "root": {"level": "INFO", "handlers": ["out"]},
})
logging.getLogger("jobs.nightly").info("backup finished")
for handler in list(logging.getLogger().handlers):
    logging.getLogger().removeHandler(handler)
    handler.close()
```

</details>